# Ensemble Techniques on Fashion MNIST dataset

In [1]:
%load_ext cuml.accel

In [5]:
import numpy 
import cv2
import tensorflow as tf

from tensorflow.keras.datasets import fashion_mnist
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import joblib

In [6]:
data = fashion_mnist.load_data()
(X_train, y_train), (X_test, y_test) = data

In [7]:
X_train = X_train / 255.0
X_test = X_test / 255.0

In [8]:
nsamples, nx, ny = X_train.shape
X_train = X_train.reshape((nsamples,nx*ny))

nsamples, nx, ny = X_test.shape
X_test = X_test.reshape((nsamples,nx*ny))

## Voting Classifier

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC

In [15]:
estimator = []

estimator.append(("LR", LogisticRegression(solver = 'saga',
                                           multi_class = 'multinomial',
                                           max_iter = 200)))
estimator.append(('KNN', KNeighborsClassifier(n_neighbors = 3)))

In [16]:
vot_soft = VotingClassifier(estimators = estimator, voting = 'hard', n_jobs = -1)
vot_soft.fit(X_train, y_train)

VotingClassifier(estimators=[('LR',
                              LogisticRegression(max_iter=200,
                                                 multi_class='multinomial',
                                                 solver='saga')),
                             ('KNN', KNeighborsClassifier(n_neighbors=3))],
                 n_jobs=-1)

In [17]:
print("Voting Classifier Accuracy: ", accuracy_score(y_test, vot_soft.predict(X_test)))
print("Voting Classifier F1 Score: ", f1_score(y_test, vot_soft.predict(X_test), average = 'weighted'))
print("Voting Classifier Precision: ", precision_score(y_test, vot_soft.predict(X_test), average = 'weighted'))
print("Voting Classifier Recall: ", recall_score(y_test, vot_soft.predict(X_test), average = 'weighted'))

Voting Classifier Accuracy:  0.8552
Voting Classifier F1 Score:  0.8514466760329426
Voting Classifier Precision:  0.8630276228998731
Voting Classifier Recall:  0.8552


In [ ]:
joblib.dump(vot_soft, 'voting_classifier_model.pkl')

## Stacking Classifier

In [2]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import StackingClassifier

In [9]:
knn = KNeighborsClassifier(n_neighbors = 3)
lr = LogisticRegression(solver = 'saga', multi_class = 'multinomial', max_iter = 200)
gnb = GaussianNB()

In [ ]:
model_stack = StackingClassifier(estimators = [('knn', knn), ('lr', lr), ('gnb', gnb)], 
                                 final_estimator = LogisticRegression(),
                                 n_jobs = -1, stack_method = 'auto')
model_stack.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
print("Stacking Classifier Accuracy: ", accuracy_score(y_test, model_stack.predict(X_test)))
print("Stacking Classifier F1 Score: ", f1_score(y_test, model_stack.predict(X_test), average = 'weighted'))
print("Stacking Classifier Precision: ", precision_score(y_test, model_stack.predict(X_test), average = 'weighted'))
print("Stacking Classifier Recall: ", recall_score(y_test, model_stack.predict(X_test), average = 'weighted'))

## AdaBoost Classifier

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

In [ ]:
adclf = AdaBoostClassifier(DecisionTreeClassifier(max_depth = 1), n_estimators = 200, learning_rate = 0.5)
adclf.fit(X_train, y_train)

In [ ]:
print("AdaBoost Classifier Accuracy: ", accuracy_score(y_test, adclf.predict(X_test)))
print("AdaBoost Classifier F1 Score: ", f1_score(y_test, adclf.predict(X_test), average = 'weighted'))
print("AdaBoost Classifier Precision: ", precision_score(y_test, adclf.predict(X_test), average = 'weighted'))
print("AdaBoost Classifier Recall: ", recall_score(y_test, adclf.predict(X_test), average = 'weighted'))

## Comparing Models

### VotingClassifier

In [ ]:
pre = X_test[0].reshape(1, -1)
y_pred_2 = vot_soft.predict(pre)

p = plt.subplot(1, 2, 1)
p.imshow(X_test[0].reshape(28, 28), cmap = plt.cm.gray, interpolation = 'bilinear')

print("Predicted Label: ", y_pred_2[0])
print("Actual Label: ", y_test[0])

### StackingClassifier

In [ ]:
pre = X_test[0].reshape(1, -1)
y_pred = model_stack.predict(pre)

p = plt.subplot(1, 2, 1)
p.imshow(X_test[0].reshape(28, 28), cmap = plt.cm.gray, interpolation = 'bilinear')

print("Predicted Label: ", y_pred[0])
print("Actual Label: ", y_test[0])

### Adaboost

In [ ]:
pre = X_test[0].reshape(1, -1)
y_pred = adclf.predict(pre)

p = plt.subplot(1, 2, 1)
p.imshow(X_test[0].reshape(28, 28), cmap = plt.cm.gray, interpolation = 'bilinear')

print("Predicted Label: ", y_pred[0])
print("Actual Label: ", y_test[0])